# Audit Cube to Bowl before training

Grade [`triton7777/cube_to_bowl_so100`](https://huggingface.co/datasets/triton7777/cube_to_bowl_so100), 10 SO-100 pick-and-place episodes, with Kalanos and see what it flags.
CPU only, no API key, nothing is published. Choose **Runtime → Run all**.

## Install

Kalanos with video checks, and two helpers, at pinned versions.

In [ ]:
# @title Install Kalanos
%pip install -q --no-warn-conflicts \
    "kalanos[numeric,video,hf]==0.10.2" huggingface-hub==1.33.0 plotly==7.1.0

## Download the dataset

About 66 MB, pinned to one revision so every run sees the same files.

In [ ]:
# @title Download the dataset
from pathlib import Path

from huggingface_hub import snapshot_download


DATASET_ID = "triton7777/cube_to_bowl_so100"
# A fixed revision keeps the files, and so every result below, the same on each run.
REVISION = "9f490c4e624034eb5c6e961e65767abb9ab7c391"

DATASET = Path(
    snapshot_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        revision=REVISION,
        local_dir="cube_to_bowl_so100",
    )
)

## Grade it

The CLI prints a report card and saves the full report as JSON: 9 of 10 episodes pass and 1 needs review.
It uses the `vision-imitation-v1` profile at the full tier, which adds camera checks.

In [ ]:
# @title Grade it with the Kalanos CLI
# `--verbosity error` hides warnings about file formats this dataset doesn't use.
# Piping through `cat` stops the CLI from animating a progress spinner in the cell,
# and `--color` keeps the card's colours through the pipe.
!kalanos --verbosity error grade {DATASET} --profile vision-imitation-v1 --tier full --report audit.json --color 2>&1 | cat

## Explore the report in Python

One row per episode. Review is a result, not an error: Kalanos found something worth a look, and the decision is yours.

In [ ]:
# @title Load the report into a table
import polars as pl

import kalanos


loaded = kalanos.load_report("audit.json")
# `load_report` also reads reports from older schemas, which have no episodes.
# Assigning after the check gives `report` that narrower type in later cells too.
assert isinstance(loaded, kalanos.Report)
report = loaded

rows = [
    {"episode": e.id, "decision": e.eligibility.status.value}
    for e in report.episodes
    # Every episode in a saved report has a decision; this only tells the type checker.
    if e.eligibility is not None
]

# Wide enough to show the episode number at the end of each id.
pl.Config.set_fmt_str_lengths(50)
pl.DataFrame(rows).sort("decision", "episode", descending=[True, False])

## What Kalanos flagged

Episode 000008 goes to review because its gripper camera is far blurrier than that camera's other episodes.
Below are frames from the same points in episode 000008 and in the camera's typical episode, then episode 000008's clip itself.
Kalanos calls this a heuristic: a reason to look, not proof of a fault.

In [ ]:
# @title Show the frames Kalanos flagged
import json

import av
import plotly.graph_objects as go
from IPython.display import Video, display
from plotly.subplots import make_subplots


# Orange marks the flagged episode; text stays in ink and muted grey.
REVIEW_COLOR = "#eb6834"
SURFACE, INK, MUTED = "#fcfcfb", "#2b2b29", "#6b6a65"
POINTS = [0.2, 0.4, 0.6, 0.8]

# The blur finding, found in the report as finding → episode and camera.
finding = next(f for f in report.findings if f.metric_id == "vision.blur_vs_camera")
camera = finding.source_field
median = finding.evidence["camera_median"]


def episode_index(episode_id):
    return int(episode_id.rsplit("_", 1)[1])


# Kalanos's blur value for every episode on that camera, found as episode → stream → metric.
blur = {
    episode_index(e.id): s.metrics["sharpness_score"].evidence["laplacian_var"]
    for e in report.episodes
    for s in e.streams
    if s.source_field == camera and "sharpness_score" in s.metrics
}
flagged = episode_index(finding.episode_id)
# The camera's typical episode is the one whose blur value sits closest to its median.
# With an even count the median falls halfway between two episodes, so the
# distance is rounded and the lower-numbered episode wins the tie.
typical = min(
    (i for i in blur if i != flagged),
    key=lambda i: (round(abs(blur[i] - median), 6), i),
)

# LeRobot v2.1 names each episode's video from a template in meta/info.json.
info = json.loads((DATASET / "meta" / "info.json").read_text())


def video_file(index):
    return DATASET / info["video_path"].format(
        episode_chunk=index // info["chunks_size"],
        video_key=camera,
        episode_index=index,
    )


def frames_at(path, points):
    """Return the frames at the given fractions of a video, as RGB arrays.

    Parameters
    ----------
    path : Path
        The video file.
    points : list of float
        Fractions of the video, from 0 to 1.

    Returns
    -------
    list of numpy.ndarray
        One ``height × width × 3`` array per fraction, in the order given.
    """
    with av.open(str(path)) as container:
        # Counting packets reads the file without decoding it.
        n = sum(1 for p in container.demux(video=0) if p.size)
    wanted = {round(p * (n - 1)): k for k, p in enumerate(points)}
    frames = [None] * len(points)
    with av.open(str(path)) as container:
        for number, frame in enumerate(container.decode(video=0)):
            if number in wanted:
                frames[wanted[number]] = frame.to_ndarray(format="rgb24")
    if any(f is None for f in frames):
        raise ValueError(f"{path} decoded fewer frames than it has packets.")
    return frames


episodes = [
    (
        flagged,
        f"<b>Episode {flagged}</b> · blur value {blur[flagged]:.1f}, sent to review",
    ),
    (
        typical,
        f"<b>Episode {typical}</b> · blur value {blur[typical]:.1f},"
        " the camera's typical episode",
    ),
]
frames = [frames_at(video_file(index), POINTS) for index, _ in episodes]

# A fixed width, and a height that fits the frames' shape, keep each row label
# right above its frames.
WIDTH, MARGIN = 960, {"t": 180, "l": 30, "r": 20, "b": 20}
H_SPACING, V_SPACING = 0.01, 0.2
frame_height, frame_width = frames[0][0].shape[:2]
cell_width = (WIDTH - MARGIN["l"] - MARGIN["r"]) * (1 - 3 * H_SPACING) / 4
plot_height = 2 * cell_width * frame_height / frame_width / (1 - V_SPACING)

fig = make_subplots(
    rows=2,
    cols=4,
    horizontal_spacing=H_SPACING,
    vertical_spacing=V_SPACING,
    column_titles=[f"{p:.0%} through" for p in POINTS],
)
for row, images in enumerate(frames, start=1):
    for col, image in enumerate(images, start=1):
        fig.add_trace(go.Image(z=image, hoverinfo="skip"), row=row, col=col)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.update_annotations(font={"color": MUTED, "size": 14}, yshift=2)

# A label above each row, and a short orange rule beside the flagged one.
for row, (_, label) in enumerate(episodes, start=1):
    top = fig.get_subplot(row, 1).yaxis.domain[1]
    fig.add_annotation(
        text=label,
        x=0,
        y=top,
        xref="paper",
        yref="paper",
        xanchor="left",
        yanchor="bottom",
        # Row 1's label sits above the column labels.
        yshift=30 if row == 1 else 6,
        showarrow=False,
        font={"color": INK, "size": 15},
    )
row_1 = fig.get_subplot(1, 1).yaxis.domain
fig.add_shape(
    type="line",
    xref="paper",
    yref="paper",
    x0=-0.012,
    x1=-0.012,
    y0=row_1[0],
    y1=row_1[1],
    line={"color": REVIEW_COLOR, "width": 4},
)

fig.update_layout(
    font={
        "family": "system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif",
        "size": 15,
        "color": INK,
    },
    paper_bgcolor=SURFACE,
    plot_bgcolor=SURFACE,
    width=WIDTH,
    height=round(MARGIN["t"] + plot_height + MARGIN["b"]),
    margin=MARGIN,
    title={
        "text": f"Episode {flagged}'s {camera.removeprefix('observation.images.')}"
        f" camera reads {blur[flagged]:.1f} on Kalanos's blur measure,"
        f"<br>against {median:.1f} for a typical episode",
        "font": {"size": 22},
        # Anchored to the frames; the padding clears the subtitle and the labels below it.
        "yref": "paper",
        "y": 1,
        "yanchor": "bottom",
        "pad": {"b": 135},
        "subtitle": {
            "text": "Higher is sharper. Frames from the same points in each episode.",
            "font": {"color": MUTED, "size": 15},
        },
    },
)
fig.show()

display(Video(str(video_file(flagged)), embed=True, width=640))

## Download the reports

The report as an HTML page and as JSON, zipped.

In [ ]:
# @title Write the HTML report
# The same grade again, written as an HTML page.
# `tail` keeps the card's last lines: the file written, or the error if it failed.
!kalanos --verbosity error grade {DATASET} --profile vision-imitation-v1 --tier full --report audit.html 2>&1 | tail -n 2

In [ ]:
# @title Download both
import zipfile

from IPython.display import FileLink, display


with zipfile.ZipFile("kalanos-reports.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write("audit.json")
    archive.write("audit.html")

try:
    # Only Colab has this module; it sends the file to the browser.
    from google.colab import files  # pyright: ignore[reportMissingImports]
except ImportError:
    display(FileLink("kalanos-reports.zip"))
else:
    files.download("kalanos-reports.zip")

**Continue with Kalanos:** [Browse dataset reports](https://hub.kalanos.ai) · [Use the grader locally](https://github.com/KalanosAI/kalanos) · [Attach an audit to a training run](https://github.com/KalanosAI/kalanos-tracking)